<a href="https://colab.research.google.com/github/sanej/operational-readiness-intelligence/blob/main/operational_readiness_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Industrial Operations Assistant: Mistral-Powered RAG

### Business Context
In high-stakes industries like Pharma, Energy, and Manufacturing, critical decisions often rely on fragmented data (SOPs, incident logs, permits). The 'bottleneck' is the manual review time required to ensure compliance and safety.

### Hypothesis
A RAG pipeline using Mistral can reduce evidence-review time and improve traceability by:
1.  **Retrieving** current evidence from siloed documents.
2.  **Citing** specific material claims for human verification.
3.  **Surfacing conflicts** (e.g., a pressure spike vs. a maintenance delay).

---

## Section 1: Environment Setup
We install the necessary libraries for Mistral API interaction, document processing (LangChain), and vector indexing (FAISS).

In [1]:
import os
import getpass
import sys
from google.colab import userdata

# 1. Handle Authentication
try:
    api_key = userdata.get('MISTRAL_API_KEY')
    os.environ['MISTRAL_API_KEY'] = api_key
    print('✅ MISTRAL_API_KEY loaded from Secrets.')
except Exception:
    print('⚠️ MISTRAL_API_KEY not found in Secrets.')
    api_key = getpass.getpass('Please paste your Mistral API Key here: ')
    os.environ['MISTRAL_API_KEY'] = api_key

# 2. Verify SDK v1.0+
try:
    from mistralai import Mistral
    import langchain_mistralai
    print('✅ SDK Status: Mistral v1.0+ is ready.')
except ImportError:
    print('❌ SDK missing. Installing necessary libraries...')
    !pip install -q mistralai==1.0.0 langchain-mistralai langchain-community faiss-cpu pypdf
    print('✨ Installation complete. Please Restart Session if errors persist.')

✅ MISTRAL_API_KEY loaded from Secrets.
✅ SDK Status: Mistral v1.0+ is ready.


In [2]:
import os
api_key = os.environ.get('MISTRAL_API_KEY')
if not api_key:
    print('❌ MISTRAL_API_KEY is missing! Please set it in Secrets or via os.environ.')
else:
    print(f'✅ API Key found (starts with: {api_key[:4]}...)')

✅ API Key found (starts with: pUGa...)


In [3]:
### Environment Validated
*Note: The environment has been optimized for Mistral SDK v1.0 and LangChain integration. Redundant reinstallation scripts have been removed to maintain focus on the RAG pipeline.*

SyntaxError: invalid syntax (2870346952.py, line 2)

In [ ]:
---

SyntaxError: invalid syntax (1947214667.py, line 1)

\## Section 2: Document Ingestion & Chunking

**Technical Choice:** We use `RecursiveCharacterTextSplitter`. Unlike simple splitting, this attempts to keep paragraphs and sentences together, preserving the semantic context necessary for industrial SOPs.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Mocked Industrial Dataset representing fragmented records
documents = [
    {"source": "SOP-702-Maintenance", "text": "Standard Operating Procedure for Chemical Reactors: All valves must be inspected every 24 hours. Pressure should not exceed 150 PSI.", "date": "2023-10-01"},
    {"source": "Incident-Log-A", "text": "On Oct 15th, Reactor 2 showed a pressure spike of 160 PSI. Maintenance was delayed by 12 hours due to permit backlog.", "date": "2023-10-15"},
    {"source": "Permit-System-v2", "text": "Hot work permits require signatures from the site manager and the safety officer. Digital logs must be updated in real-time.", "date": "2024-01-10"}
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

processed_chunks = []
metadatas = []

for doc in documents:
    chunks = text_splitter.split_text(doc['text'])
    for chunk in chunks:
        processed_chunks.append(chunk)
        metadatas.append({"source": doc['source'], "date": doc['date']})

print(f"Ingested {len(documents)} documents into {len(processed_chunks)} chunks.")

Ingested 3 documents into 3 chunks.


In [7]:
from langchain_community.document_loaders import PyPDFLoader

pdf_files = [
    '/content/PTW-2026-0412-work-permit.pdf',
    '/content/INSP-2026-003-inspection-finding.pdf'
]

pdf_chunks = []
pdf_metadatas = []

for pdf_path in pdf_files:
    try:
        loader = PyPDFLoader(pdf_path)
        data = loader.load()
        # Split the loaded PDF pages
        docs = text_splitter.split_documents(data)
        for doc in docs:
            pdf_chunks.append(doc.page_content)
            # Preserve file source in metadata
            pdf_metadatas.append({"source": pdf_path.split('/')[-1], "date": "2026-04-12"})
        print(f"✅ Successfully processed {pdf_path}")
    except Exception as e:
        print(f"❌ Could not process {pdf_path}: {e}")

# Extend our main processing lists
processed_chunks.extend(pdf_chunks)
metadatas.extend(pdf_metadatas)

print(f"\nTotal chunks in pipeline: {len(processed_chunks)}")

✅ Successfully processed /content/PTW-2026-0412-work-permit.pdf
✅ Successfully processed /content/INSP-2026-003-inspection-finding.pdf

Total chunks in pipeline: 13


## Section 3: Vector Embeddings & Storage

**Technical Choice:** We use `FAISS` (Facebook AI Similarity Search) for our vector store. It is lightweight, efficient for CPU usage, and scales well to millions of vectors for production deployments.

In [5]:
import os
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings
from mistralai import Mistral

# Ensure api_key is pulled from environment
api_key = os.environ.get('MISTRAL_API_KEY')

if not api_key:
    raise ValueError('MISTRAL_API_KEY is not set. Please run the setup cell in Section 1.')

# Initialize Mistral Client (SDK v1.0 syntax)
client = Mistral(api_key=api_key)

# Initialize Mistral Embeddings
embeddings = MistralAIEmbeddings(mistral_api_key=api_key)

# Create the vector database
vector_db = FAISS.from_texts(processed_chunks, embeddings, metadatas=metadatas)

print('✅ Vector store and Mistral client successfully initialized.')

/tmp/ipykernel_48838/2092583166.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Vector store and Mistral client successfully initialized.


In [8]:
# Re-initialize the vector store to include the PDF content
vector_db = FAISS.from_texts(processed_chunks, embeddings, metadatas=metadatas)
print("✅ Vector store updated with PDF content.")

✅ Vector store updated with PDF content.


## Section 4: Grounded Generation (RAG)

This function performs the 'Retrieve' and 'Generate' steps.

**Business Value:** By using a grounded prompt, we ensure the model does not hallucinate safety procedures. If the answer isn't in the provided evidence, the model is instructed to admit it.

In [6]:
def ask_industrial_assistant(query):
    if 'vector_db' not in globals():
        return "Error: Vector database not initialized."

    # 1. Similarity Search: Increase k to ensure we pull from both mock data and PDF
    search_results = vector_db.similarity_search(query, k=6)

    # 2. Build Context String with Metadata
    context = "\n\n".join([
        f"[Source: {res.metadata.get('source', 'Unknown')}]\nContent: {res.page_content}"
        for res in search_results
    ])

    # 3. Grounded System Prompt
    prompt = f"""You are a Technical Operations Assistant.
    Answer the question using ONLY the context provided below.

    Context:
    {context}

    Question: {query}
    """

    # 4. Generate with Mistral SDK v1.0
    chat_response = client.chat.complete(
        model="mistral-large-latest",
        messages=[{"role": "user", "content": prompt}]
    )
    return chat_response.choices[0].message.content

# FINAL DEMO: Comparing SOP limits (Mock) vs Logs (Mock) + Inspection (PDF)
query = "What is the pressure limit in SOP-702 and was it violated in Incident-Log-A?"
print(f"Query: {query}\n")
try:
    response = ask_industrial_assistant(query)
    print(f"Assistant Response:\n{response}")
except Exception as e:
    print(f"❌ Error: {e}")

Query: What is the pressure limit in SOP-702 and was it violated in Incident-Log-A?

Assistant Response:
The pressure limit in **SOP-702-Maintenance** is **150 PSI**.

In **Incident-Log-A**, Reactor 2 experienced a pressure spike of **160 PSI**, which **violated** the SOP-702 pressure limit.


In [9]:
# Test query specifically for PDF content
pdf_query = "What are the key findings in the 2026 inspection report?"
print(f"Query: {pdf_query}\n")
print(ask_industrial_assistant(pdf_query))

Query: What are the key findings in the 2026 inspection report?

Based on the provided context, the key findings in the **INSP-2026-003 inspection report** are as follows:

1. **Concurrent Effective Revisions of SOP-CL-004**:
   - Two active revisions of **SOP-CL-004** (Rev 3, effective 2025-08-12, and Rev 4, effective 2026-03-02) were found in the controlled document system.
   - Neither revision was marked as **superseded**, creating ambiguity in which version should be followed.

2. **Material Differences in Cleaning Requirements**:
   - The two revisions specify **different cleaning parameters** for **GRN-2100**, including:
     - **Maximum clean hold time**: 72 hours (Rev 3) vs. 120 hours (Rev 4).
     - **Swab sampling locations**: 6 (Rev 3) vs. 4 (Rev 4).
   - This raises uncertainty about which requirements applied to cleaning operations after **2026-03-02**.

3. **Classification & Status**:
   - The finding is classified as **Major**.
   - The associated **CAPA-2026-019** is *

## Section 5: Implementation Summary & Production Scaling

### Current Implementation Status
We have successfully built a functional end-to-end RAG pipeline with the following capabilities:
1.  **Hybrid Data Ingestion**: The system processes structured text (mock logs) and unstructured files (`PTW-2026-0412` and `INSP-2026-003` PDFs).
2.  **Semantic Chunking**: Used `RecursiveCharacterTextSplitter` to maintain technical context across document boundaries.
3.  **Vectorized Retrieval**: Integrated Mistral AI Embeddings with a FAISS vector store for high-accuracy similarity search.
4.  **Grounded Reasoning**: Leveraged `mistral-large-latest` to identify complex compliance gaps (e.g., SOP pressure limit violations and conflicting document revisions).

### Future Scaling for Production
To transition this prototype into an enterprise-grade industrial assistant, the following steps are recommended:

1.  **Scaling**: Transition from FAISS in-memory to a managed vector DB (e.g., Pinecone, Milvus, or pgvector) to handle millions of documents.
2.  **Data Governance**: Implement Role-Based Access Control (RBAC) so users only retrieve documents they are authorized to see.
3.  **Advanced Evaluation**: Use frameworks like **RAGAS** to measure 'Faithfulness' (hallucination check) and 'Relevancy' (search quality) against a golden dataset.
4.  **Real-world Deployment**: Integrate with existing Document Management Systems (DMS) via webhooks to update the vector store automatically whenever a new SOP or Incident Log is signed.